# Notebook 03 - Regression Model and Metrics Revision

This notebook revises regression concepts using a house price prediction dataset.

Algorithms covered:

1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor

Metrics covered:

- MAE
- MSE
- RMSE
- R² score

In [ ]:
# Core libraries used throughout the Day 1 notebooks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Display settings make notebook output easier to read during training
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
house_df = pd.read_csv('../datasets/house_price_training.csv')
house_df.head()

In [ ]:
X = house_df.drop(columns=['property_id', 'price_inr'])
y = house_df['price_inr']

numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print('Numerical:', numerical_features)
print('Categorical:', categorical_features)

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

In [ ]:
def evaluate_regression_model(model_name, model, X_test, y_test):
    """Evaluate regression model and return common regression metrics."""
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print('==============================')
    print(model_name)
    print('==============================')
    print('MAE :', round(mae, 2))
    print('MSE :', round(mse, 2))
    print('RMSE:', round(rmse, 2))
    print('R2  :', round(r2, 4))

    return {
        'model_name': model_name,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'r2_score': r2
    }

## 1. Linear Regression

Simple idea: It tries to fit a straight-line style relationship between features and target.

Good baseline model for regression.

In [ ]:
linear_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

linear_pipeline.fit(X_train, y_train)
linear_metrics = evaluate_regression_model('Linear Regression', linear_pipeline, X_test, y_test)

## 2. Decision Tree Regressor

Simple idea: It splits data into branches and predicts average target value in each final region.

In [ ]:
tree_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(max_depth=6, random_state=42))
])

tree_pipeline.fit(X_train, y_train)
tree_metrics = evaluate_regression_model('Decision Tree Regressor', tree_pipeline, X_test, y_test)

## 3. Random Forest Regressor

Simple idea: It trains many decision trees and averages their predictions.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=150, max_depth=8, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_metrics = evaluate_regression_model('Random Forest Regressor', rf_pipeline, X_test, y_test)

In [ ]:
regression_results = pd.DataFrame([
    linear_metrics,
    tree_metrics,
    rf_metrics
]).sort_values(by='rmse')

regression_results

In [ ]:
regression_results.to_csv('../outputs/regression_model_comparison.csv', index=False)
print('Saved regression comparison file to ../outputs/regression_model_comparison.csv')